# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:

# Load datasets
aviation_df = pd.read_csv("AviationData.csv", encoding = "latin1", low_memory=False)
state_codes_df = pd.read_csv("USState_Codes.csv")





In [3]:
#inspect the aviation data
print("Aviation dataset shape:", aviation_df.shape)
print(aviation_df.info())
print(aviation_df.isna().sum())
print(aviation_df.describe(include='all'))

Aviation dataset shape: (88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Ma

In [4]:
#inspecting us codes data
print("\nUS State Codes dataset shape:", state_codes_df.shape)
print(state_codes_df.info())
print(state_codes_df.isna().sum())
print(state_codes_df.describe(include='all'))


US State Codes dataset shape: (62, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   US_State      62 non-null     object
 1   Abbreviation  62 non-null     object
dtypes: object(2)
memory usage: 1.1+ KB
None
US_State        0
Abbreviation    0
dtype: int64
       US_State Abbreviation
count        62           62
unique       62           62
top     Alabama           AL
freq          1            1


In [5]:
#import the data and the 1st 5 rows
df = pd.read_csv("USState_Codes.csv", encoding="latin1")
df.head()
#overview of the data
df.info()
#null values per column
df.isna().sum()
# for mean, std deviation, mim, max and quartiles
df.describe()
#check for duplicating
df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   US_State      62 non-null     object
 1   Abbreviation  62 non-null     object
dtypes: object(2)
memory usage: 1.1+ KB


0

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
#inspect relevant columns
print(aviation_df['Make'].value_counts().head(20))
print(aviation_df['Model'].value_counts().head(20))
print(aviation_df['Amateur.Built'].value_counts())
print(aviation_df['Aircraft.damage'].value_counts())

Make
Cessna               22227
Piper                12029
CESSNA                4922
Beech                 4330
PIPER                 2841
Bell                  2134
Boeing                1594
BOEING                1151
Grumman               1094
Mooney                1092
BEECH                 1042
Robinson               946
Bellanca               886
Hughes                 795
Schweizer              629
Air Tractor            595
BELL                   588
Mcdonnell Douglas      526
Aeronca                487
Maule                  445
Name: count, dtype: int64
Model
152          2367
172          1756
172N         1164
PA-28-140     932
150           829
172M          798
172P          689
182           659
180           622
150M          585
PA-18         581
PA-18-150     578
PA-28-180     572
PA-28-161     569
PA-28-181     532
206B          524
737           489
PA-38-112     469
150L          461
G-164A        460
Name: count, dtype: int64
Amateur.Built
No     80312
Yes     84

In [7]:
#reasonable imputations
aviation_df['Amateur.Built'] = aviation_df['Amateur.Built'].fillna("No")
aviation_df['Aircraft.damage'] = aviation_df['Aircraft.damage'].fillna("Unknown")
aviation_df['Number.of.Engines'] = aviation_df['Number.of.Engines'].fillna(1)
aviation_df['Engine.Type'] = aviation_df['Engine.Type'].fillna("Unknown")


In [8]:
# Convert Event.Date to datetime
aviation_df['Event.Date'] = pd.to_datetime(
    aviation_df['Event.Date'], errors='coerce', infer_datetime_format=True
)

# Convert Publication.Date too (optional)
aviation_df['Publication.Date'] = pd.to_datetime(
    aviation_df['Publication.Date'], errors='coerce', infer_datetime_format=True
)

# Check conversion
print(aviation_df[['Event.Date','Publication.Date']].head())
print(aviation_df['Event.Date'].dtype)


  Event.Date Publication.Date
0 1948-10-24              NaT
1 1962-07-19       1996-09-19
2 1974-08-30       2007-02-26
3 1977-06-19       2000-09-12
4 1979-08-02       1980-04-16
datetime64[ns]


C:\Users\Admin\AppData\Local\Temp\ipykernel_23084\2719224208.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  aviation_df['Event.Date'] = pd.to_datetime(
C:\Users\Admin\AppData\Local\Temp\ipykernel_23084\2719224208.py:7: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  aviation_df['Publication.Date'] = pd.to_datetime(
C:\Users\Admin\AppData\Local\Temp\ipykernel_23084\2719224208.py:7: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  aviation_df[

In [9]:
#filtering for the required date range and exclude amatuer aircraft

aviation_df = aviation_df[
    (aviation_df['Event.Date'].dt.year >= 1983) &
    (aviation_df['Event.Date'].dt.year <= 2023)
]
aviation_df = aviation_df[aviation_df['Amateur.Built'].str.upper() == "NO"]

In [10]:
#keep relevant columns
relevant_cols = [
    'Event.Date','Location','Country','Make','Model','Aircraft.damage',
    'Total.Fatal.Injuries','Total.Serious.Injuries','Total.Minor.Injuries',
    'Total.Uninjured','Number.of.Engines','Engine.Type','Broad.phase.of.flight',
    'Weather.Condition'
]
aviation_df = aviation_df[relevant_cols]


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [11]:
# injury columns
injury_cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

# Assumption: Missing values mean "not reported" therefore equal to 0
aviation_df[injury_cols] = aviation_df[injury_cols].fillna(0)

#aircraft damage
# Assumption: null damages mean unknown
aviation_df['Aircraft.damage'] = aviation_df['Aircraft.damage'].fillna("Unknown")


In [12]:
# Estimate total occupants as sum of all injury categories
aviation_df['Total.Occupants'] = (
    aviation_df['Total.Fatal.Injuries'] +
    aviation_df['Total.Serious.Injuries'] +
    aviation_df['Total.Minor.Injuries'] +
    aviation_df['Total.Uninjured']
)

# Assumption: If all injury counts are missing, Total.Occupants = 0
aviation_df['Total.Occupants'] = aviation_df['Total.Occupants'].fillna(0)


In [13]:
# Fraction of fatalities per accident
aviation_df['FatalRate'] = aviation_df['Total.Fatal.Injuries'] / aviation_df['Total.Occupants']

# Fraction of serious injuries per accident
aviation_df['SeriousRate'] = aviation_df['Total.Serious.Injuries'] / aviation_df['Total.Occupants']

# Combined severe injury metric (fatal + serious)
aviation_df['SevereInjuryRate'] = (
    (aviation_df['Total.Fatal.Injuries'] + aviation_df['Total.Serious.Injuries']) 
    / aviation_df['Total.Occupants']
)

# Replace NaN (from division by zero occupants) with 0
aviation_df[['FatalRate','SeriousRate','SevereInjuryRate']] = aviation_df[['FatalRate','SeriousRate','SevereInjuryRate']].fillna(0)

# --- Robustness to destruction ---
# Binary indicator: 1 if aircraft destroyed, 0 otherwise
aviation_df['Destroyed'] = aviation_df['Aircraft.damage'].str.upper().str.contains("DESTROYED").astype(int)


In [14]:
print(aviation_df[['Make','Model','Total.Occupants','FatalRate','SeriousRate','SevereInjuryRate','Destroyed']].head(10))


                Make       Model  Total.Occupants  FatalRate  SeriousRate  \
3600         Piccard        AX-6              2.0        0.0          0.5   
3601          Cessna        182P              4.0        0.0          0.0   
3602          Cessna       182RG              2.0        0.0          0.0   
3603          Cessna        182P              1.0        0.0          0.0   
3604           Piper  PA-28R-200              2.0        0.0          0.0   
3605          Cessna         140              2.0        0.0          0.0   
3606   Balloon Works  FIREFLY 7B              2.0        0.0          0.5   
3607          Cessna        340A              4.0        0.0          0.0   
3608  North American        T-6G              2.0        1.0          0.0   
3609           Piper   PA-24-250              3.0        1.0          0.0   

      SevereInjuryRate  Destroyed  
3600               0.5          0  
3601               0.0          0  
3602               0.0          0  
3603    

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [15]:
# Inspect unique values
print(aviation_df['Aircraft.damage'].value_counts(dropna=False))


Aircraft.damage
Substantial    55715
Destroyed      15503
Unknown         3226
Minor           2617
Name: count, dtype: int64


In [16]:
# Binary destruction indicator
aviation_df['Destroyed'] = aviation_df['Aircraft.damage'].apply(
    lambda x: 1 if "DESTROYED" in str(x) else 0
)


In [17]:
#output
print(aviation_df[['Aircraft.damage','Destroyed']].head(20))
print("Proportion destroyed:", aviation_df['Destroyed'].mean())


     Aircraft.damage  Destroyed
3600         Unknown          0
3601     Substantial          0
3602     Substantial          0
3603     Substantial          0
3604     Substantial          0
3605     Substantial          0
3606         Unknown          0
3607       Destroyed          0
3608       Destroyed          0
3609       Destroyed          0
3610     Substantial          0
3611     Substantial          0
3612     Substantial          0
3613     Substantial          0
3614     Substantial          0
3615     Substantial          0
3616     Substantial          0
3617     Substantial          0
3618       Destroyed          0
3619       Destroyed          0
Proportion destroyed: 0.0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [18]:
# 1. Standardize casing and trim whitespace
aviation_df['Make'] = aviation_df['Make'].str.strip().str.upper()

# 2. Drop rows with missing Make
aviation_df = aviation_df.dropna(subset=['Make'])

# 3. Unify common variants
make_replacements = {
    "MCDONNELL DOUGLAS": "MCDONNELL DOUGLAS",
    "DOUGLAS": "MCDONNELL DOUGLAS",
    "CESSNA AIRCRAFT": "CESSNA",
    "BEECH": "BEECHCRAFT",
    "BEECH AIRCRAFT": "BEECHCRAFT",
    "LOCKHEED MARTIN": "LOCKHEED",
    "PIPER AIRCRAFT": "PIPER"
}
aviation_df['Make'] = aviation_df['Make'].replace(make_replacements)

# 4. Filter by frequency threshold
make_counts = aviation_df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
aviation_df = aviation_df[aviation_df['Make'].isin(valid_makes)]

# Check results
print("Remaining manufacturers:", aviation_df['Make'].unique())
print("Counts per manufacturer:\n", aviation_df['Make'].value_counts())


Remaining manufacturers: ['CESSNA' 'PIPER' 'BALLOON WORKS' 'NORTH AMERICAN' 'BEECHCRAFT'
 'SWEARINGEN' 'CANADAIR' 'MCDONNELL DOUGLAS' 'MOONEY' 'BELL' 'SCHWEIZER'
 'BOEING' 'HILLER' 'BELLANCA' 'ROBINSON' 'GRUMMAN' 'LUSCOMBE' 'ENSTROM'
 'LOCKHEED' 'CHAMPION' 'GRUMMAN AMERICAN' 'MAULE' 'AERO COMMANDER'
 'MITSUBISHI' 'STINSON' 'SMITH, TED AEROSTAR' 'GATES LEARJET' 'ROCKWELL'
 'TAYLORCRAFT' 'GULFSTREAM' 'EMBRAER' 'AERONCA' 'NAVION' 'AEROSPATIALE'
 'HELIO' 'DE HAVILLAND' 'HUGHES' 'AYRES' 'RAVEN' 'RYAN' 'BURKHART GROB'
 'AMERICAN' 'AIR TRACTOR' 'LET' 'ERCOUPE (ENG & RESEARCH CORP.)'
 'GREAT LAKES' 'WEATHERLY' 'GLOBE' 'SCHEMPP-HIRTH' 'WACO' 'SCHLEICHER'
 'LAKE' 'GRUMMAN-SCHWEIZER' 'BOEING STEARMAN' 'LEARJET' 'PITTS' 'MBB'
 'SIKORSKY' 'FAIRCHILD' 'WSK PZL MIELEC' 'AEROSTAR' 'PILATUS'
 'AIRBUS INDUSTRIE' 'CAMERON' 'FOKKER' 'BRITISH AEROSPACE' 'AGUSTA'
 'CIRRUS' 'SOCATA' 'CIRRUS DESIGN CORP.' 'AVIAT' 'EUROCOPTER'
 'ROCKWELL INTERNATIONAL' 'ERCOUPE' 'AIRBUS' 'BOMBARDIER' 'DEHAVILLAND'
 'RAYTHEON A

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [19]:
#dropping rows where model is missing
aviation_df = aviation_df.dropna(subset = ['Model'])
# inspecting model/make by looking at most common models
print(aviation_df['Model'].value_counts().head(20))
# Inspect combinations of Make + Model, whether there is an overlap
print(aviation_df.groupby('Model')['Make'].nunique().sort_values(ascending=False).head(20))


Model
152          2229
172          1651
172N         1093
PA-28-140     865
172M          759
150           752
172P          665
182           617
180           596
150M          551
PA-18         550
PA-18-150     549
PA-28-180     547
PA-28-161     523
PA-28-181     509
737           488
206B          488
150L          435
A36           430
PA-38-112     425
Name: count, dtype: int64
Model
G-164A    7
500       7
G-164B    6
AA-5      5
G-164     5
AA-5B     5
G-164C    5
S-2R      4
700       4
112       4
8KCAB     4
100       4
7ECA      4
7EC       4
500-S     4
690C      4
G-1159    4
690B      4
690A      4
300       4
Name: Make, dtype: int64


In [20]:
#creating a make and model key 
aviation_df['Make_Model'] = aviation_df['Make'].str.upper().str.strip() + "_" + aviation_df['Model'].str.upper().str.strip()


In [21]:
print(aviation_df[['Make','Model','Make_Model']].head(20))
print("Unique Make_Model combinations:", aviation_df['Make_Model'].nunique())


                Make        Model                Make_Model
3601          CESSNA         182P               CESSNA_182P
3602          CESSNA        182RG              CESSNA_182RG
3603          CESSNA         182P               CESSNA_182P
3604           PIPER   PA-28R-200          PIPER_PA-28R-200
3605          CESSNA          140                CESSNA_140
3606   BALLOON WORKS   FIREFLY 7B  BALLOON WORKS_FIREFLY 7B
3607          CESSNA         340A               CESSNA_340A
3608  NORTH AMERICAN         T-6G       NORTH AMERICAN_T-6G
3609           PIPER    PA-24-250           PIPER_PA-24-250
3610           PIPER   PA-32-301R          PIPER_PA-32-301R
3611      BEECHCRAFT        V-35B          BEECHCRAFT_V-35B
3612      BEECHCRAFT          A36            BEECHCRAFT_A36
3613          CESSNA          180                CESSNA_180
3614          CESSNA          170                CESSNA_170
3615          CESSNA          172                CESSNA_172
3616      SWEARINGEN     SA-226TC       

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [36]:
# Add Purpose.of.flight back into your working dataframe
aviation_raw = pd.read_csv("AviationData.csv", encoding = 'latin1')
aviation_df['Purpose.of.flight'] = aviation_raw.loc[aviation_df.index, 'Purpose.of.flight']

# Clean it
aviation_df['Purpose.of.flight'] = aviation_df['Purpose.of.flight'].str.strip().str.upper()
aviation_df['Purpose.of.flight'] = aviation_df['Purpose.of.flight'].fillna("UNKNOWN")


C:\Users\Admin\AppData\Local\Temp\ipykernel_23084\52088301.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_raw = pd.read_csv("AviationData.csv", encoding = 'latin1')


In [38]:
# engine type
aviation_df['Engine.Type'] = aviation_df['Engine.Type'].str.strip().str.upper()
aviation_df['Engine.Type'] = aviation_df['Engine.Type'].replace({
    "TURBO JET": "TURBOJET",
    "TURBO PROP": "TURBOPROP",
    "RECIPROCATING": "RECIPROCATING"
})
aviation_df['Engine.Type'] = aviation_df['Engine.Type'].fillna("UNKNOWN")

# weather condition
aviation_df['Weather.Condition'] = aviation_df['Weather.Condition'].str.strip().str.upper()
aviation_df['Weather.Condition'] = aviation_df['Weather.Condition'].fillna("UNKNOWN")

# number of engines
aviation_df['Number.of.Engines'] = aviation_df['Number.of.Engines'].fillna(1).astype(int)

#purpose of flight
aviation_df['Purpose.of.flight'] = aviation_df['Purpose.of.flight'].str.strip().str.upper()
aviation_df['Purpose.of.flight'] = aviation_df['Purpose.of.flight'].fillna("UNKNOWN")

#broad phase of flight
aviation_df['Broad.phase.of.flight'] = aviation_df['Broad.phase.of.flight'].str.strip().str.upper()
aviation_df['Broad.phase.of.flight'] = aviation_df['Broad.phase.of.flight'].fillna("UNKNOWN")


In [32]:
print(aviation_df.columns.tolist())
print(state_codes_df.columns.tolist())

['Event.Date', 'Location', 'Country', 'Make', 'Model', 'Aircraft.damage', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Number.of.Engines', 'Engine.Type', 'Broad.phase.of.flight', 'Weather.Condition', 'Total.Occupants', 'FatalRate', 'SeriousRate', 'SevereInjuryRate', 'Destroyed', 'Make_Model']
['US_State', 'Abbreviation']


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [33]:
#inspect missing values
missing_counts = aviation_df.isna().sum()
missing_ratio = missing_counts / len(aviation_df)
print(missing_ratio.sort_values(ascending=False))
# Drop columns with greater than 40% NaNs
aviation_df = aviation_df.dropna(axis=1, thresh=len(aviation_df)*0.4)


Broad.phase.of.flight     0.290576
Country                   0.002748
Location                  0.000680
Event.Date                0.000000
Engine.Type               0.000000
Destroyed                 0.000000
SevereInjuryRate          0.000000
SeriousRate               0.000000
FatalRate                 0.000000
Total.Occupants           0.000000
Weather.Condition         0.000000
Number.of.Engines         0.000000
Total.Uninjured           0.000000
Total.Minor.Injuries      0.000000
Total.Serious.Injuries    0.000000
Total.Fatal.Injuries      0.000000
Aircraft.damage           0.000000
Model                     0.000000
Make                      0.000000
Make_Model                0.000000
dtype: float64


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
# Save cleaned dataframe to CSV
aviation_df.to_csv("AviationData_Cleaned.csv",encoding = "latin1", index=False)